## Importação das bibliotecas
Nesta etapa, importamos as bibliotecas necessárias para a manipulação, 
análise e visualização dos dados dos pedidos de manutenção.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

**Justificação das bibliotecas utilizadas:**

- **Pandas (pd):** permite carregar, manipular e analisar
  dados tabulares.

- **NumPy (np):** permite realizar operações numéricas
  e matemáticas.

- **Matplotlib (plt):** permite construir gráficos para
  a análise exploratória dos dados.

## 2. Carregamento dos dados de treino

Nesta etapa, carregamos o conjunto de dados de treino
a partir do ficheiro treino.csv.

O objetivo é conhecer a estrutura inicial dos dados
e os atributos que caracterizam os pedidos de manutenção.

Nesta fase, utilizamos exclusivamente os dados de treino,
reservando os dados de validação e de teste para as
etapas posteriores do projeto.

In [ ]:
dados_treino = pd.read_csv("../dados/treino.csv")
dados_treino.head()

In [ ]:
dados_treino.info()

In [ ]:
dados_treino.shape

**Justificação:**

A utilização da função head() permite observar uma
amostra inicial dos dados, identificar os atributos
disponíveis e verificar se o ficheiro foi carregado
corretamente.

A separação dos conjuntos de treino, validação e teste
permite prevenir a fuga de informação (Data Leakage).

## 3. Auditoria dos dados

### 3.1. Identificação dos valores em falta

O objetivo desta análise é identificar os atributos
que apresentam valores em falta, quantificar a sua
ocorrência e avaliar a necessidade do seu tratamento.

In [ ]:
valores_em_falta = dados_treino.isnull().sum()
percentagem_em_falta = (valores_em_falta / len(dados_treino)) * 100

auditoria_faltas = pd.DataFrame({
    "Tipo de dados": dados_treino.dtypes,
    "Valores em falta": valores_em_falta,
    "Percentagem (%)": percentagem_em_falta.round(2)
})

auditoria_faltas

**Justificação e interpretação dos resultados**

A análise dos 1800 pedidos de manutenção permitiu identificar os tipos de dados e quantificar 
os valores em falta em cada atributo do conjunto de treino.

Foram identificados cinco atributos com valores em falta:

- `utilizadores_afetados`: 71 registos (3,94%).
- `alternativa_disponivel`: 73 registos (4,06%).
- `horas_desde_deteccao`: 115 registos (6,39%).
- `reincidencias_30d`: 68 registos (3,78%).
- `idade_anos`: 112 registos (6,22%).

O atributo `horas_desde_deteccao` apresenta a maior percentagem de valores em falta, seguido de `idade_anos`.
Os restantes sete atributos não apresentam valores em falta, incluindo a variável-alvo `urgente` e o identificador `id_pedido`.

Observou-se também que os atributos `utilizadores_afetados` e `reincidencias_30d`, apesar de representarem contagens
inteiras, foram interpretados pelo Pandas como float64 devido à presença de valores em falta.

Nesta fase, decidiu-se preservar os dados originais, sem eliminar registos nem substituir os valores em falta,
uma vez que a ausência de informação não representa necessariamente um valor zero ou uma resposta negativa.

O tratamento destes valores será realizado na fase de preparação dos dados, considerando o tipo e a natureza
de cada atributo, utilizando apenas os dados de treino para ajustar os métodos de imputação durante a comparação
dos modelos, evitando assim a fuga de informação (Data Leakage).

### 3.2. Identificação de duplicados

O objetivo desta etapa é verificar a existência de
identificadores repetidos e registos totalmente duplicados,
de modo a garantir a integridade e a qualidade dos dados
do conjunto de treino.

In [ ]:
ids_duplicados = dados_treino["id_pedido"].duplicated().sum()

print("Número de identificadores duplicados:", ids_duplicados)

**Justificação - Identificadores duplicados**

Foi verificada a existência de identificadores repetidos na coluna `id_pedido`. A análise não identificou duplicados (0 ocorrências em 1 800 registos).

**Decisão:** manter todos os registos. O identificador será utilizado para rastreabilidade e análise de erros, mas não será utilizado como atributo de entrada dos modelos, pois não representa informação relevante para prever a urgência.

In [ ]:
# Contar linhas totalmente duplicadas
linhas_duplicadas = dados_treino.duplicated().sum()

print("Número de linhas totalmente duplicadas:", linhas_duplicadas)

**Observação — Linhas totalmente duplicadas**

A análise dos 1800 registos do conjunto de treino não
identificou linhas totalmente duplicadas, tendo sido
obtidas zero ocorrências.

**Decisão:** manter todos os registos, uma vez que não
foram encontradas duplicações completas que justificassem
a sua eliminação.

Pedidos com identificadores diferentes podem apresentar
características semelhantes sem representarem necessariamente
o mesmo pedido de manutenção.

**Impacto:** nenhum registo foi eliminado, mantendo-se
os 1800 pedidos do conjunto de treino.

### 3.3. Verificação dos domínios dos atributos categóricos

O objetivo desta etapa é identificar as categorias existentes
nos atributos categóricos do conjunto de treino e verificar
se os valores observados correspondem aos domínios esperados.

Esta análise permite identificar categorias inesperadas,
inconsistências e possíveis erros de preenchimento dos dados.

In [ ]:
# Mostrar as categorias existentes em cada atributo

print("Equipamentos:")
print(sorted(dados_treino["equipamento"].dropna().unique()))

print("\nContextos:")
print(sorted(dados_treino["contexto"].dropna().unique()))

print("\nPeríodos:")
print(sorted(dados_treino["periodo"].dropna().unique()))

print("\nInterrupção total:")
print(sorted(dados_treino["interrupcao_total"].dropna().unique()))

print("\nAlternativa disponível:")
print(sorted(dados_treino["alternativa_disponivel"].dropna().unique()))

print("\nSinal de segurança:")
print(sorted(dados_treino["sinal_seguranca"].dropna().unique()))

print("\nUrgência:")
print(sorted(dados_treino["urgente"].dropna().unique()))


**Justificação e interpretação dos resultados**

A análise dos domínios dos atributos categóricos permitiu
confirmar que todos os valores observados pertencem aos
domínios esperados.

Foram identificadas cinco categorias de equipamentos,
quatro categorias de contextos e dois períodos de atividade.

Os atributos interrupcao_total, alternativa_disponivel
e sinal_seguranca apresentam exclusivamente as categorias
"sim" e "nao" entre os valores observados.

A variável-alvo urgente apresenta exclusivamente os valores
0 e 1, correspondentes às classes não urgente e urgente,
respetivamente.

Não foram identificadas categorias inesperadas, erros
de escrita ou valores fora dos domínios esperados.

**Decisão:** preservar todas as categorias originais,
sem realizar correções ou substituições nesta etapa.
Os valores em falta serão tratados posteriormente.

**Justificação:** os valores observados encontram-se
em conformidade com os domínios esperados, não existindo
evidências que justifiquem a sua alteração.

**Impacto:** zero registos alterados ou eliminados,
mantendo-se os 1800 pedidos do conjunto de treino.

### 3.4. Verificação dos domínios e unidades dos atributos numéricos

O objetivo desta etapa é verificar se os valores dos
atributos numéricos respeitam os domínios esperados,
considerando as respetivas unidades de medida.

Pretende-se identificar valores impossíveis ou incoerentes
que possam comprometer a qualidade dos dados e o desempenho
dos modelos de classificação.

In [ ]:
# Definir os atributos numéricos
atributos_numericos = [
    "utilizadores_afetados",
    "horas_desde_deteccao",
    "reincidencias_30d",
    "idade_anos"
]

# Construir uma tabela com os valores mínimos e máximos
auditoria_numerica = pd.DataFrame({
    "Mínimo observado": dados_treino[atributos_numericos].min(),
    "Máximo observado": dados_treino[atributos_numericos].max(),
    "Mínimo esperado": [1, 0.1, 0, 0.1],
    "Máximo esperado": [240, 72, 5, 20],
    "Unidade": ["Pessoas", "Horas", "Ocorrências", "Anos"]
})

# Apresentar os resultados
auditoria_numerica

In [ ]:
# Verificar valores fracionários nos atributos de contagem
atributos_contagem = [
    "utilizadores_afetados",
    "reincidencias_30d"
]

for atributo in atributos_contagem:
    valores = dados_treino[atributo].dropna()
    invalidos = (valores % 1 != 0).sum()

    print(f"{atributo}: {invalidos} valores fracionários")

**Justificação e interpretação dos resultados**

A análise dos atributos numéricos permitiu verificar
que todos os valores observados se encontram dentro
dos limites esperados.

O atributo utilizadores_afetados apresenta valores
entre 1 e 240 pessoas, enquanto reincidencias_30d
apresenta valores entre 0 e 5 ocorrências.

O atributo idade_anos apresenta valores entre 0,1
e 20 anos, respeitando o domínio esperado.

Relativamente ao atributo horas_desde_deteccao,
o valor máximo observado foi de 51,6 horas, inferior
ao limite máximo permitido de 72 horas. Esta diferença
não representa uma inconsistência, uma vez que o valor
observado pertence ao intervalo admissível.

A verificação complementar dos atributos de contagem
não identificou valores fracionários.

**Decisão:** manter os valores numéricos originais,
sem realizar correções ou eliminações nesta etapa.

**Justificação:** não foram identificados valores fora
dos limites esperados nem valores fracionários
nos atributos de contagem.

**Impacto:** zero registos alterados ou eliminados,
preservando-se os 1800 pedidos do conjunto de treino.

### 3.5. Identificação de valores atípicos (Outliers)

O objetivo desta etapa é identificar potenciais valores
atípicos nos atributos numéricos do conjunto de treino,
utilizando o método do intervalo interquartil (IQR).

Pretende-se distinguir valores estatisticamente atípicos
de valores inválidos, evitando a eliminação injustificada
de pedidos de manutenção que possam representar situações
reais e relevantes para a classificação da urgência.

In [ ]:
# Calcular o primeiro e o terceiro quartil
Q1 = dados_treino[atributos_numericos].quantile(0.25)
Q3 = dados_treino[atributos_numericos].quantile(0.75)

# Calcular o intervalo interquartil
IQR = Q3 - Q1

# Calcular os limites estatísticos
limite_inferior = Q1 - 1.5 * IQR
limite_superior = Q3 + 1.5 * IQR

# Identificar os valores atípicos
outliers = (
    (dados_treino[atributos_numericos] < limite_inferior) |
    (dados_treino[atributos_numericos] > limite_superior)
)

# Organizar os resultados numa tabela
auditoria_outliers = pd.DataFrame({
    "Q1": Q1,
    "Q3": Q3,
    "IQR": IQR,
    "Limite inferior": limite_inferior,
    "Limite superior": limite_superior,
    "Valores atípicos": outliers.sum()
})

# Apresentar os resultados
auditoria_outliers.round(2)

**Justificação e interpretação dos resultados**

A aplicação do método do intervalo interquartil (IQR) permitiu identificar valores atípicos em três dos quatro
atributos numéricos do conjunto de treino.

O atributo utilizadores_afetados apresenta 120 valores atípicos, sendo o atributo com maior número de ocorrências.
Estes valores podem representar pedidos de manutenção que afetam um número elevado de utilizadores.

O atributo horas_desde_deteccao apresenta 59 valores atípicos, correspondentes a pedidos com tempos de
deteção decorridos superiores a 26,1 horas.

O atributo idade_anos apresenta 62 valores atípicos, correspondentes a equipamentos com idade superior
a 13,15 anos.

O atributo reincidencias_30d não apresenta valores atípicos segundo o método utilizado.

No total, foram identificadas 241 ocorrências de valores atípicos, não correspondendo necessariamente a 241
pedidos distintos.

**Decisão:** manter todos os valores atípicos identificados.

**Justificação:** a verificação anterior confirmou que os valores numéricos respeitam os domínios esperados.
Assim, os valores atípicos podem representar situações válidas e relevantes para a classificação da urgência.

A sua eliminação poderia provocar perda de informação e reduzir a capacidade dos modelos para identificar
situações menos frequentes.

**Impacto:** nenhum registo foi alterado ou eliminado, mantendo-se os 1800 pedidos do conjunto de treino.

### 3.6. Consolidação da tabela de auditoria

O objetivo desta etapa é consolidar os resultados obtidos
durante a auditoria do conjunto de treino.

A tabela de auditoria reúne os tipos de dados, as unidades
de medida, os domínios esperados, a quantidade e a percentagem
de valores em falta e os potenciais valores atípicos.

Esta consolidação permite obter uma visão global da qualidade
dos dados e fundamentar as decisões que serão tomadas na
fase de preparação dos dados.

In [ ]:
# Definir os domínios esperados
dominios = {
    "id_pedido": "Identificador único",
    "equipamento": "computador, rede, projetor, climatizacao, eletrico",
    "contexto": "aula, laboratorio, servicos, espaco_comum",
    "utilizadores_afetados": "Inteiros entre 1 e 240",
    "interrupcao_total": "sim, nao",
    "alternativa_disponivel": "sim, nao",
    "horas_desde_deteccao": "0.1 a 72",
    "reincidencias_30d": "Inteiros entre 0 e 5",
    "idade_anos": "0.1 a 20",
    "sinal_seguranca": "sim, nao",
    "periodo": "letivo, nao_letivo",
    "urgente": "0, 1"
}

# Definir as unidades de medida
unidades = {
    "id_pedido": "Não aplicável",
    "equipamento": "Não aplicável",
    "contexto": "Não aplicável",
    "utilizadores_afetados": "Pessoas",
    "interrupcao_total": "Não aplicável",
    "alternativa_disponivel": "Não aplicável",
    "horas_desde_deteccao": "Horas",
    "reincidencias_30d": "Ocorrências",
    "idade_anos": "Anos",
    "sinal_seguranca": "Não aplicável",
    "periodo": "Não aplicável",
    "urgente": "Não aplicável"
}



In [ ]:
# Reutilizar a tabela de valores em falta da secção 3.1
auditoria_final = auditoria_faltas.copy()

# Adicionar as unidades de medida
auditoria_final["Unidade"] = auditoria_final.index.map(unidades)

# Adicionar os domínios esperados
auditoria_final["Domínio esperado"] = auditoria_final.index.map(dominios)

# Adicionar os valores atípicos calculados na secção 3.5
auditoria_final["Valores atípicos"] = (
    auditoria_outliers["Valores atípicos"]
    .reindex(auditoria_final.index)
    .astype("Int64")
)

# Reorganizar as colunas
auditoria_final = auditoria_final[[
    "Tipo de dados",
    "Unidade",
    "Domínio esperado",
    "Valores em falta",
    "Percentagem (%)",
    "Valores atípicos"
]]

pd.set_option("display.max_colwidth", None)
auditoria_final

#### Síntese das decisões de limpeza

| Problema analisado | Evidência | Decisão | Impacto |
|---|---|---|---|
| Valores em falta | 439 valores em falta distribuídos por cinco atributos | Adiar o tratamento para a fase de preparação dos dados | Nenhum registo eliminado |
| Identificadores duplicados | 0 ocorrências | Manter os identificadores originais | Nenhum registo alterado |
| Linhas duplicadas | 0 ocorrências | Manter todos os registos | Nenhum registo eliminado |
| Categorias inesperadas | 0 ocorrências | Preservar as categorias originais | Nenhum registo alterado |
| Valores numéricos inválidos | 0 ocorrências | Manter os valores originais | Nenhum registo alterado |
| Valores fracionários em atributos de contagem | 0 ocorrências | Manter os valores originais | Nenhum registo alterado |
| Valores atípicos | 241 ocorrências em três atributos | Preservar os valores atípicos válidos | Nenhum registo eliminado |

**Justificação e conclusão da auditoria**

A auditoria permitiu avaliar a estrutura, a integridade
e a qualidade dos 1800 pedidos de manutenção presentes
no conjunto de treino.

Foram identificados 439 valores em falta, distribuídos
por cinco atributos, e 241 ocorrências de valores
estatisticamente atípicos.

Estas ocorrências não correspondem necessariamente ao
mesmo número de pedidos distintos, uma vez que um pedido
pode apresentar várias ocorrências.

Os valores atípicos foram preservados por respeitarem
os domínios esperados e poderem representar situações
relevantes para a classificação da urgência.

O tratamento dos valores em falta será realizado na fase
de preparação dos dados, através de métodos de imputação
adequados à natureza de cada atributo.

Durante a auditoria, não foram realizadas alterações
aos dados originais, preservando-se os 1800 registos
do conjunto de treino.

## 4. Análise exploratória e visualização dos dados

### 4.1. Percentagem de valores em falta por atributo

**Objetivo:**

Representar graficamente a percentagem de valores em falta
em cada atributo do conjunto de treino.

**Pergunta de análise:**

Quais são os atributos que apresentam maior percentagem
de valores em falta e que necessitam de tratamento na
fase de preparação dos dados?

In [ ]:
# Criar a figura
plt.figure(figsize=(12, 6))

# Construir o gráfico de barras
plt.bar(
    percentagem_em_falta.index,
    percentagem_em_falta.values,
    color="steelblue",
    edgecolor="black"
)

# Definir o título e os nomes dos eixos
plt.title("Percentagem de valores em falta por atributo")
plt.xlabel("Atributos")
plt.ylabel("Percentagem de valores em falta (%)")

# Definir os limites do eixo vertical
plt.ylim(0, 8)

# Rodar os nomes dos atributos
plt.xticks(rotation=45, ha="right")

# Apresentar a percentagem acima de cada barra
for i, valor in enumerate(percentagem_em_falta.values):
    plt.text(i, valor + 0.1, f"{valor:.2f}%", ha="center")

# Adicionar linhas de referência
plt.grid(axis="y", linestyle="--", alpha=0.4)

# Ajustar a apresentação do gráfico
plt.tight_layout()

# Apresentar o gráfico
plt.show()

**Justificação da escolha do gráfico**

Foi utilizado um gráfico de barras por permitir comparar a percentagem de valores em falta entre os diferentes
atributos do conjunto de treino.

O eixo horizontal representa os atributos e o eixo vertical representa a percentagem de valores em falta, calculada
relativamente aos 1800 pedidos de manutenção.

O eixo vertical inicia-se em zero para evitar distorções na comparação visual das percentagens.

**Interpretação dos resultados**

O atributo horas_desde_deteccao apresenta a maior percentagem de valores em falta (6,39%), seguido de
idade_anos (6,22%).

Os atributos alternativa_disponivel, utilizadores_afetados
e reincidencias_30d apresentam percentagens de 4,06%,
3,94% e 3,78%, respetivamente.

Os restantes sete atributos não apresentam valores em falta.

Esta análise permite identificar os cinco atributos que
necessitam de tratamento na fase de preparação dos dados.

**Limitações**

O gráfico permite identificar a quantidade relativa de valores em falta, mas não permite determinar as causas
da ausência de informação, nem o seu impacto no desempenho dos modelos de classificação.

Também não permite identificar se os valores em falta ocorrem simultaneamente nos mesmos pedidos.

**Decisão**

Preservar os dados originais e realizar o tratamento dos valores em falta posteriormente, através de métodos
de imputação adequados a cada tipo de atributo.

### 4.2. Distribuição dos pedidos de manutenção por equipamento

**Objetivo:**

Analisar a distribuição dos pedidos de manutenção pelas
diferentes categorias de equipamentos presentes no
conjunto de treino.

**Pergunta de análise:**

Quais são os tipos de equipamentos que apresentam maior
e menor número de pedidos de manutenção?

A análise permite compreender a representatividade de
cada categoria no conjunto de dados.

In [ ]:
# Calcular a frequência absoluta por equipamento
frequencia_absoluta = dados_treino["equipamento"].value_counts()

# Calcular a frequência relativa em percentagem
frequencia_relativa = (
    frequencia_absoluta / len(dados_treino)
) * 100

# Organizar os resultados numa tabela
distribuicao_equipamentos = pd.DataFrame({
    "Número de pedidos": frequencia_absoluta,
    "Percentagem (%)": frequencia_relativa.round(2)
})

# Apresentar a tabela
distribuicao_equipamentos

In [ ]:
# Criar a figura
plt.figure(figsize=(10, 6))

# Construir o gráfico de barras
plt.bar(
    frequencia_absoluta.index,
    frequencia_absoluta.values,
    color="steelblue",
    edgecolor="black"
)

# Definir o título e os nomes dos eixos
plt.title("Distribuição dos pedidos de manutenção por equipamento")
plt.xlabel("Tipo de equipamento")
plt.ylabel("Número de pedidos")

# Definir os limites do eixo vertical
plt.ylim(0, frequencia_absoluta.max() * 1.15)

# Apresentar a quantidade e a percentagem acima de cada barra
for i, quantidade in enumerate(frequencia_absoluta.values):

    percentagem = frequencia_relativa.iloc[i]

    plt.text(
        i,
        quantidade + 5,
        f"{quantidade} ({percentagem:.2f}%)",
        ha="center"
    )

# Adicionar linhas de referência
plt.grid(axis="y", linestyle="--", alpha=0.4)

# Ajustar a apresentação
plt.tight_layout()

# Apresentar o gráfico
plt.show()


**Justificação da escolha do gráfico**

Foi utilizado um gráfico de barras por permitir comparar as frequências absolutas entre as diferentes categorias
de equipamentos.

O eixo horizontal representa os tipos de equipamentos, enquanto o eixo vertical representa o número de pedidos
de manutenção.

As etiquetas acima das barras apresentam a frequência absoluta e a frequência relativa de cada categoria.

As percentagens foram calculadas relativamente aos 1800 pedidos do conjunto de treino.

Todos os pedidos foram considerados nesta análise, uma vez que o atributo equipamento não apresenta
valores em falta.

O eixo vertical inicia-se em zero, garantindo uma
comparação visual adequada entre as categorias.

**Limitações**

O gráfico permite analisar a distribuição dos pedidos por equipamento, mas não permite determinar quais
os equipamentos que apresentam maior probabilidade de gerar pedidos urgentes.

Também não permite concluir quais os equipamentos que avariam mais frequentemente, uma vez que não
dispomos de informação sobre a quantidade total de equipamentos existentes no campus.

**Interpretação dos resultados**

A análise da distribuição dos pedidos de manutenção permitiu identificar diferenças na representatividade
das cinco categorias de equipamentos.

A categoria computador apresenta o maior número de pedidos, com 561 ocorrências (31,17%), seguida da
categoria rede, com 415 ocorrências (23,06%).

Em conjunto, estas duas categorias representam 54,22% dos pedidos de manutenção do conjunto de treino.

As categorias climatizacao e projetor apresentam frequências semelhantes, com 328 pedidos (18,22%)
e 327 pedidos (18,17%), respetivamente.

A categoria eletrico apresenta a menor frequência, com 169 pedidos, correspondendo a 9,39% do total.

Esta distribuição demonstra que as categorias não estão representadas de forma uniforme no conjunto
de treino.

Contudo, uma maior frequência de pedidos não significa
necessariamente uma maior probabilidade de urgência.

Esta relação será analisada posteriormente através da comparação da proporção de pedidos urgentes
entre as categorias de equipamentos.

**Decisão**

Manter todas as categorias e todos os registos, preservando a distribuição original do conjunto
de treino.

### 4.3. Distribuição antes e depois da imputação pela mediana

**Objetivo:**

Analisar o impacto da imputação pela mediana na
distribuição do atributo horas_desde_deteccao.

**Pergunta de análise:**

De que forma a substituição dos valores em falta pela
mediana altera a distribuição estatística do atributo?

**Escolha do atributo:**

Foi selecionado o atributo horas_desde_deteccao por
apresentar a maior percentagem de valores em falta
(6,39%) e conter 59 valores estatisticamente atípicos.

A experiência será realizada exclusivamente sobre uma
cópia dos dados de treino, preservando os dados originais.